# UCL Data Transformation

**Install and Imports** 

In [0]:
%pip install azure-storage-blob python-dotenv 

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
import io
import pandas as pd
from datetime import datetime
from azure.storage.blob import BlobServiceClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, to_timestamp, trim, when, lit, min as spark_min, max as spark_max, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, TimestampType

**Configuration**

In [0]:
STORAGE_ACCOUNT_NAME= "Azure Storage account name" #Replace it with your azure storage account name
STORAGE_ACCOUNT_KEY= "Azure Storage account key" #Replace it with your azure storage account key

#Azure Storage account name
BRONZE_CONTAINER = "bronze"
SILVER_CONTAINER = "silver"

conn_str = f"DefaultEndpointsProtocol=https;AccountName={STORAGE_ACCOUNT_NAME};AccountKey={STORAGE_ACCOUNT_KEY};EndpointSuffix=core.windows.net"
try:
    az_client = BlobServiceClient.from_connection_string(conn_str)
    print("--Connected to Azure Storage--")
except Exception as e:
    print("Connection failed: ", e)

--Connected to Azure Storage--


**Helper Functions**

In [0]:
# Return list with all the blobs inside the container
def list_bronze_blobs(prefix: str) -> list:
    """Return list of blob names matching prefix in Bronze container."""
    container_client = az_client.get_container_client(BRONZE_CONTAINER)
    blobs = [
        b.name
        for b in container_client.list_blobs(name_starts_with=prefix)
    ]
    print(f"Found {len(blobs)} blobs with prefix '{prefix}':")
    for b in blobs:
        print(f"  {b}")
    return blobs

# Download blobs from container
def download_bronze_json(blob_path: str) -> dict | list | None:
    """
    Download a JSON file from Bronze container.
    Returns parsed Python object or None if failed.
    """
    try:
        blob_client = az_client.get_blob_client(
            container = BRONZE_CONTAINER,
            blob      = blob_path,
        )
        raw_bytes = blob_client.download_blob().readall()
        data      = json.loads(raw_bytes)
        size_kb   = len(raw_bytes) / 1024
        print(f"Downloaded: {blob_path} ({size_kb:.1f} KB)")
        return data
    except Exception as e:
        print(f"Failed to download {blob_path}: {e}")
        return None    

# Helper method that list all the blobs in the bronze container
def upload_silver_parquet(df, blob_path: str) -> bool:
    """ Upload Spark DataFrame to Azure Silver as Parquet. """
    try:
        print(f"Collecting {df.count()} rows...")

        # Using collect method convert dataframe into python object and cast as Dict.
        rows_as_dicts = [row.asDict(recursive=True) for row in df.collect()]
        print(f"Converted {len(rows_as_dicts)} rows to plain dicts")

        # creating pandas dataframe by dict
        pandas_df = pd.DataFrame(rows_as_dicts)
        print(f"Pandas shape: {pandas_df.shape}")

        # Serialise to Parquet bytes in memory
        buffer = io.BytesIO()
        pandas_df.to_parquet(
            buffer,
            engine      = "pyarrow",
            index       = False,
            compression = "snappy",
        )
        buffer.seek(0)
        parquet_bytes = buffer.getvalue()
        size_kb = len(parquet_bytes) / 1024
        print(f"Parquet serialised: {size_kb:.1f} KB")

        # Upload to Azure Silver
        blob_client = az_client.get_blob_client(container = SILVER_CONTAINER, blob = blob_path,)
        blob_client.upload_blob(parquet_bytes, overwrite=True)
        print(f"Uploaded to Silver: {SILVER_CONTAINER}/{blob_path}")
        return True

    except MemoryError:
        print("MemoryError: DataFrame too large.")
        return False

    except Exception as e:
        print(f" Upload failed: {e}")
        return False

print("--Helper functions defined--")


--Helper functions defined--


**Read Bronze + Explore raw data**


In [0]:

bronze_blobs = list_bronze_blobs(prefix="matches/")

latest_blob = sorted(bronze_blobs)[-1]
print(f"\nUsing latest file: {latest_blob}")

raw_data = download_bronze_json(latest_blob)

if not raw_data:
    raise ValueError("Failed to load Bronze match data — check Azure connection")

matches_raw = raw_data.get("matches", [])
metadata    = raw_data.get("_ingestion_metadata", {})

print(f"\n--- Bronze File Summary ---")
print(f"Ingested at  : {metadata.get('ingested_at', 'unknown')}")
print(f"Source       : {metadata.get('source', 'unknown')}")
print(f"Match count  : {len(matches_raw)}")

# Show structure of first match to understand the schema
if matches_raw:
    print(f"\n--- Raw match keys (first record) ---")
    for key, val in matches_raw[0].items():
        print(f"  {key}: {type(val).__name__} = {str(val)[:70]}")


Found 1 blobs with prefix 'matches/':
  matches/2026-05-16_ucl_matches_season_2024.json

Using latest file: matches/2026-05-16_ucl_matches_season_2024.json
Downloaded: matches/2026-05-16_ucl_matches_season_2024.json (303.9 KB)

--- Bronze File Summary ---
Ingested at  : 2026-05-16T00:07:10.505989
Source       : football-data.org
Match count  : 189

--- Raw match keys (first record) ---
  area: dict = {'id': 2077, 'name': 'Europe', 'code': 'EUR', 'flag': 'https://crests.
  competition: dict = {'id': 2001, 'name': 'UEFA Champions League', 'code': 'CL', 'type': 'C
  season: dict = {'id': 2350, 'startDate': '2024-09-17', 'endDate': '2025-05-31', 'curr
  id: int = 524039
  utcDate: str = 2024-09-17T16:45:00Z
  status: str = FINISHED
  matchday: int = 1
  stage: str = LEAGUE_STAGE
  group: NoneType = None
  lastUpdated: str = 2025-06-14T20:20:19Z
  homeTeam: dict = {'id': 1871, 'name': 'BSC Young Boys', 'shortName': 'Young Boys', 'tla
  awayTeam: dict = {'id': 58, 'name': 'Aston Villa FC', '

**PySpark Transformation (JSON -> DataFrame)**

In [0]:
# Function to flatten one row of API response.
def flatten_match(data:dict) -> dict:
    "This method extract necessary fields from nested dictionary and return a flat dictionary"

    # Score is nested: {"fullTime": {"home": 2, "away": 1}}
    score = data.get('score', {})
    full_time = score.get('fullTime', {})
    half_time = score.get('halfTime', {})

    home_team = data.get('homeTeam',{})
    away_team = data.get('awayTeam',{})

    competition = data.get('competition',{})

    return {
        
        # Match Identifiers
        "match_id" : data.get("id"),
        "utc_date" : data.get("utcDate"),
        "status"   : data.get("status"),
        "matchday" : data.get("matchday"),
        "stage"    : data.get("stage"),
        "group_name" : data.get("group") or "NA",

        # Competition
        "competition_id"    : competition.get("id"),
        "competition_name"  : competition.get("name"),
        "competition_code"  : competition.get("code"),
        "season_id"         : data.get("season", {}).get("id"),

        # Teams
        "home_team_id"      : home_team.get("id"),
        "home_team_name"    : home_team.get("name"),
        "home_team_short"   : home_team.get("shortName"),
        "away_team_id"      : away_team.get("id"),
        "away_team_name"    : away_team.get("name"),
        "away_team_short"   : away_team.get("shortName"),

        # Scores
        "home_score_ft"     : full_time.get("home"),
        "away_score_ft"     : full_time.get("away"),
        "home_score_ht"     : half_time.get("home"),
        "away_score_ht"     : half_time.get("away"),

        #Winner
        "winner"            : score.get("winner"),

        #refree
        "referee_name"      : (
            data.get("referees", [{}])[0].get("name")
            if data.get("referees") else None)
    }

flattened = [flatten_match(m) for m in matches_raw]
print(f"=== Flattened : {len(flattened)} match records ===")

df_bronze = spark.createDataFrame(flattened)

print(f"\n--- Bronze DataFrame Schema ---")
df_bronze.printSchema()

# Applying Transformation 

df_silver = (
    df_bronze
    .withColumn("match_id", col("match_id").cast(IntegerType()))
    .withColumn("matchday", col("matchday").cast(IntegerType()))
    .withColumn("home_score_ft", col("home_score_ft").cast(IntegerType()))
    .withColumn("away_score_ft", col("away_score_ft").cast(IntegerType()))
    .withColumn("home_score_ht", col("home_score_ht").cast(IntegerType()))
    .withColumn("away_score_ht", col("away_score_ht").cast(IntegerType()))
    .withColumn("competition_id",col("competition_id").cast(IntegerType()))
    .withColumn("season_id",col("season_id").cast(IntegerType()))
    .withColumn("away_team_id",col("away_team_id").cast(IntegerType()))
    .withColumn("home_team_id",col("home_team_id").cast(IntegerType()))

    # Parse Date and time
    .withColumn("match_date", to_date(col("utc_date")))
    .withColumn("match_timestamp", to_timestamp(col("utc_date")))

    # Trimming whitespaces from string fields
    .withColumn("home_team_name", trim(col("home_team_name")))
    .withColumn("away_team_name", trim(col("away_team_name")))
    .withColumn("status", trim(col("status")))
    .withColumn("stage", trim(col("stage")))
    .withColumn("winner", trim(col("winner")))
    .withColumn("away_team_short", trim(col("away_team_short")))
    .withColumn("home_team_short", trim(col("home_team_short")))
    .withColumn("competition_code", trim(col("competition_code")))
    .withColumn("competition_name", trim(col("competition_name")))
    .withColumn("refree_name", trim(col("referee_name")))

    .withColumn("is_finished", when(col("status") == "FINISHED", True).otherwise(False))
    .withColumn("total_goals", when(col("is_finished"),col("home_score_ft") + col("away_score_ft")).otherwise(lit(None)).cast(IntegerType()))
    .withColumn("goal_difference", when(col("is_finished"),col("home_score_ft") - col("away_score_ft")).otherwise(lit(None)).cast(IntegerType()))
    .withColumn("home_result", 
                when(col("winner") == "HOME_TEAM", lit("W"))
                .when(col("winner") == "AWAY_TEAM", lit("L"))
                .when(col("winner") == "DRAW", lit("D"))
                .otherwise(lit(None))
                )
    # Adding Metadata for silver
    .withColumn("silver_processed_at",lit(current_timestamp().cast(StringType())))
    .withColumn("source_blob", lit(latest_blob))
     # drop utc column
    .drop("utc_date")
)

print("\n---Schema of Silver DataFrame---")
df_silver.printSchema()

df_silver = (
    df_silver.select(
        "match_id",
        "match_date",
        "match_timestamp",
        "status",
        "is_finished",
        "matchday",
        "stage",
        "group_name",
        "competition_id",
        "competition_name",
        "competition_code",
        "season_id",
        "home_team_id",
        "home_team_name",
        "home_team_short",
        "away_team_id",
        "away_team_name",
        "away_team_short",
        "home_score_ft",
        "away_score_ft",
        "home_score_ht",
        "away_score_ht",
        "winner",
        "home_result",
        "total_goals",
        "goal_difference",
        "referee_name",
        "silver_processed_at",
        "source_blob",
    )
)

print(f"\n--- Silver sample (5 rows, Real madrid matches only) ---")
df_silver.filter(col("home_team_name") == "Real Madrid CF").select( "match_date", "matchday", "stage",
        "home_team_name", "home_score_ft",
        "away_score_ft", "away_team_name",
        "home_result", "total_goals").limit(5).show()


=== Flattened : 189 match records ===

--- Bronze DataFrame Schema ---
root
 |-- away_score_ft: long (nullable = true)
 |-- away_score_ht: long (nullable = true)
 |-- away_team_id: long (nullable = true)
 |-- away_team_name: string (nullable = true)
 |-- away_team_short: string (nullable = true)
 |-- competition_code: string (nullable = true)
 |-- competition_id: long (nullable = true)
 |-- competition_name: string (nullable = true)
 |-- group_name: string (nullable = true)
 |-- home_score_ft: long (nullable = true)
 |-- home_score_ht: long (nullable = true)
 |-- home_team_id: long (nullable = true)
 |-- home_team_name: string (nullable = true)
 |-- home_team_short: string (nullable = true)
 |-- match_id: long (nullable = true)
 |-- matchday: long (nullable = true)
 |-- referee_name: string (nullable = true)
 |-- season_id: long (nullable = true)
 |-- stage: string (nullable = true)
 |-- status: string (nullable = true)
 |-- utc_date: string (nullable = true)
 |-- winner: string (nulla

**Data Quality Checks**

In [0]:
print("Running data quality checks on Silver DataFrame...")
print("=" * 60)

dq_passed = True
dq_results = {}

# Row Count > 0 check
row_count = df_silver.count()
check_name = "row_count_gt_zero"
passed = row_count > 0
dq_results[check_name] = {"passed": passed, "value": row_count}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | rows={row_count}")
if not passed:
    dq_passed = False


# Null Checking in Valuable columns
critical_cols = ["match_id", "match_date", "home_team_name", "away_team_name"]
for col_name in critical_cols:
    null_count = df_silver.filter(col(col_name).isNull()).count()
    check_name = f"no_nulls_{col_name}"
    passed = null_count == 0
    dq_results[check_name] = {"passed": passed, "value": null_count}
    status = "PASS" if passed else "FAIL"
    print(f"{status} | {check_name} | nulls={null_count}")
    if not passed:
        dq_passed = False


# Unique match ID check
total_rows    = df_silver.count()
distinct_ids  = df_silver.select("match_id").distinct().count()
check_name    = "match_id_unique"
passed        = total_rows == distinct_ids
dq_results[check_name] = {
    "passed": passed,
    "value": f"{distinct_ids}/{total_rows} unique"
}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | distinct={distinct_ids} total={total_rows}")
if not passed:
    dq_passed = False


# Status columns checked
known_statuses = {"FINISHED", "SCHEDULED", "POSTPONED", "CANCELLED", "IN_PLAY", "TIMED"}
actual_statuses = {
    row["status"]
    for row in df_silver.select("status").distinct().collect()
    if row["status"] is not None
}
unknown = actual_statuses - known_statuses
check_name = "status_values_known"
passed = len(unknown) == 0
dq_results[check_name] = {
    "passed": passed,
    "value": f"unknown={unknown}" if unknown else "all known"
}
status = "PASS" if passed else "WARN"
print(f"{status} | {check_name} | {dq_results[check_name]['value']}")

# Checking finished matches that don't have scores
finished_without_score = (
    df_silver
    .filter(col("is_finished"))
    .filter(col("home_score_ft").isNull())
    .count()
)
check_name = "finished_matches_have_scores"
passed     = finished_without_score == 0
dq_results[check_name] = {"passed": passed, "value": finished_without_score}
status = "PASS" if passed else "FAIL"
print(f"{status} | {check_name} | finished_without_score={finished_without_score}")
if not passed:
    dq_passed = False


# Check date range
date_range = (
    df_silver
    .filter(col("match_date").isNotNull())
    .agg(
        spark_min("match_date").alias("earliest"),
        spark_max("match_date").alias("latest"),
    )
    .collect()[0]
)
print(
    f"INFO | date_range | "
    f"earliest={date_range['earliest']} | "
    f"latest={date_range['latest']}"
)

# Final Verdict
print("=" * 60)
if dq_passed:
    print(f"ALL DATA QUALITY CHECKS PASSED — proceeding to write Silver")
else:
    failed = [k for k, v in dq_results.items() if not v["passed"]]
    print(f"DATA QUALITY FAILED — checks: {failed}")
    raise ValueError(
        f"Silver write blocked by failing DQ checks: {failed}. "
        f"Fix the data before proceeding."
    )

Running data quality checks on Silver DataFrame...
PASS | row_count_gt_zero | rows=189
PASS | no_nulls_match_id | nulls=0
PASS | no_nulls_match_date | nulls=0
PASS | no_nulls_home_team_name | nulls=0
PASS | no_nulls_away_team_name | nulls=0
PASS | match_id_unique | distinct=189 total=189
PASS | status_values_known | all known
PASS | finished_matches_have_scores | finished_without_score=0
INFO | date_range | earliest=2024-09-17 | latest=2025-05-31
ALL DATA QUALITY CHECKS PASSED — proceeding to write Silver


### Write Silver Parquet to Azure

In [0]:
# Write Silver DataFrame to Azure ADLS Gen2 

today           = datetime.now().strftime("%Y-%m-%d")
silver_blob     = f"matches/silver_ucl_matches_{today}.parquet"

print(f"Writing Silver to: {SILVER_CONTAINER}/{silver_blob}")
print(f"Row count: {df_silver.count()}")

success = upload_silver_parquet(df_silver, silver_blob)

if not success:
    raise RuntimeError("Silver write failed — check Azure credentials and /tmp/ path")

#Verify by reading back
blob_client = az_client.get_blob_client(container = SILVER_CONTAINER, blob = silver_blob)

written_size_kb = blob_client.get_blob_properties().size / 1024
print(f"\n Silver write verified")
print(f"   Path    : {SILVER_CONTAINER}/{silver_blob}")
print(f"   Size    : {written_size_kb:.1f} KB")
print(f"   Rows    : {df_silver.count()}")
print(f"   Columns : {len(df_silver.columns)}")
print(f"   Written : {datetime.now().isoformat()}")

# ── Final display — show clean Silver data ────────────────────
print("\n--- Silver UCL Matches — final output ---")
(
    df_silver
    .filter(col("is_finished"))
    .select(
        "match_date", "matchday", "stage",
        "home_team_name", "home_score_ft",
        "away_score_ft", "away_team_name",
        "home_result", "total_goals",
    )
    .orderBy("match_date", "matchday")
    .show(10, truncate=False)
)

Writing Silver to: silver/matches/silver_ucl_matches_2026-06-03.parquet
Row count: 189
Converted 189 rows to plain dicts
Pandas shape: (189, 29)
Parquet serialised: 24.1 KB
Uploaded to Silver: silver/matches/silver_ucl_matches_2026-06-03.parquet

✅ Silver write verified
   Path    : silver/matches/silver_ucl_matches_2026-06-03.parquet
   Size    : 24.1 KB
   Rows    : 189
   Columns : 29
   Written : 2026-06-03T11:27:18.367807

--- Silver UCL Matches — final output ---
+----------+--------+------------+--------------------------+-------------+-------------+------------------------+-----------+-----------+
|match_date|matchday|stage       |home_team_name            |home_score_ft|away_score_ft|away_team_name          |home_result|total_goals|
+----------+--------+------------+--------------------------+-------------+-------------+------------------------+-----------+-----------+
|2024-09-17|1       |LEAGUE_STAGE|BSC Young Boys            |0            |3            |Aston Villa FC      